Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_gru_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 6)
Las dimensiones de testX son:  (10533, 12, 6)
Las dimensiones de valX son:  (5189, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 29s - 127ms/step - ia: 0.5064 - loss: 0.4636 - mae: 0.5374 - rmse: 0.6648 - smape: 1.1390 - val_ia: 0.3764 - val_loss: 0.3617 - val_mae: 0.5190 - val_rmse: 0.5688 - val_smape: 0.9602

Epoch 2/128                                           

231/231 - 6s - 27ms/step - ia: 0.6965 - loss: 0.2564 - mae: 0.3960 - rmse: 0.5019 - smape: 0.7950 - val_ia: 0.4594 - val_loss: 0.1876 - val_mae: 0.3621 - val_rmse: 0.4095 - val_smape: 0.7962

Epoch 3/128                                           

231/231 - 7s - 30ms/step - ia: 0.7327 - loss: 0.2128 - mae: 0.3546 - rmse: 0.4565 - smape: 0.7251 - val_ia: 0.5119 - val_loss: 0.1179 - val_mae: 0.2855 - val_rmse: 0.3276 - val_smape: 0.6841

Epoch 4/128                                           

231/231 - 7s - 30ms/step - ia: 0.7527 - loss: 0.1869 - mae: 0.3312 - rmse: 0.4272 - smape: 0.6744 - val_ia: 0.5393 - val_loss: 0.0960 - val_mae: 0.2594 - val_rmse: 0.2974 - val_smape: 0.6417

Epoch 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

29/29 - 51s - 2s/step - ia: 0.3404 - loss: 0.5503 - mae: 0.5997 - rmse: 0.7277 - smape: 1.3325 - val_ia: 0.5952 - val_loss: 0.3386 - val_mae: 0.5132 - val_rmse: 0.5668 - val_smape: 0.9042

Epoch 2/16                                                                           

29/29 - 5s - 163ms/step - ia: 0.8367 - loss: 0.1029 - mae: 0.2392 - rmse: 0.3159 - smape: 0.5220 - val_ia: 0.8189 - val_loss: 0.0800 - val_mae: 0.2207 - val_rmse: 0.2804 - val_smape: 0.4839

Epoch 3/16                                                                           

29/29 - 6s - 198ms/step - ia: 0.9113 - loss: 0.0347 - mae: 0.1303 - rmse: 0.1843 - smape: 0.3295 - val_ia: 0.9025 - val_loss: 0.0307 - val_mae: 0.1267 - val_rmse: 0.1744 - val_smape: 0.3142

Epoch 4/16                                                                           

29/29 - 5s - 179ms/step - ia: 0.9394 - loss: 0.0178 - mae: 0.0897 - rmse: 0.1322 

...

116/116 - 3s - 25ms/step - ia: 0.1683 - loss: 0.8770 - mae: 0.7767 - rmse: 0.9322 - smape: 1.6709 - val_ia: 0.2854 - val_loss: 0.9391 - val_mae: 0.8047 - val_rmse: 0.9074 - val_smape: 1.6188

Epoch 2/16                                                                             

116/116 - 1s - 6ms/step - ia: 0.2763 - loss: 0.6786 - mae: 0.6786 - rmse: 0.8215 - smape: 1.5031 - val_ia: 0.3478 - val_loss: 0.7351 - val_mae: 0.7151 - val_rmse: 0.8003 - val_smape: 1.3999

Epoch 3/16                                                                             

116/116 - 1s - 6ms/step - ia: 0.4111 - loss: 0.5130 - mae: 0.5845 - rmse: 0.7120 - smape: 1.2876 - val_ia: 0.4108 - val_loss: 0.5725 - val_mae: 0.6353 - val_rmse: 0.7054 - val_smape: 1.2413

Epoch 4/16                                                                             

116/116 - 1s - 6ms/step - ia: 0.5246 - loss: 0.3875 - mae: 0.5015 - rmse: 0.6194 - smape: 1.0994 - val_ia: 0.4643 - val_loss: 0.4497 - val_mae: 0.5679 - val_rm

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



922/922 - 11s - 12ms/step - ia: 0.9356 - loss: 0.0145 - mae: 0.0825 - rmse: 0.1039 - smape: 0.2183 - val_ia: 0.6282 - val_loss: 0.0059 - val_mae: 0.0573 - val_rmse: 0.0670 - val_smape: 0.1699

Epoch 2/128                                                                           

922/922 - 7s - 8ms/step - ia: 0.9533 - loss: 0.0065 - mae: 0.0604 - rmse: 0.0758 - smape: 0.1680 - val_ia: 0.5904 - val_loss: 0.0054 - val_mae: 0.0576 - val_rmse: 0.0671 - val_smape: 0.1443

Epoch 3/128                                                                           

922/922 - 7s - 8ms/step - ia: 0.9546 - loss: 0.0061 - mae: 0.0590 - rmse: 0.0740 - smape: 0.1629 - val_ia: 0.5648 - val_loss: 0.0073 - val_mae: 0.0671 - val_rmse: 0.0764 - val_smape: 0.1409

Epoch 4/128                                                                           

922/922 - 7s - 8ms/step - ia: 0.9565 - loss: 0.0057 - mae: 0.0567 - rmse: 0.0712 - smape: 0.1543 - val_ia: 0.5058 - val_loss: 0.0103 - val_mae: 0.0840 - val_rmse

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



922/922 - 6s - 6ms/step - ia: 0.7885 - loss: 0.1352 - mae: 0.2637 - rmse: 0.3324 - smape: 0.5607 - val_ia: 0.5415 - val_loss: 0.0099 - val_mae: 0.0758 - val_rmse: 0.0859 - val_smape: 0.1962

Epoch 2/128                                                                            

922/922 - 3s - 4ms/step - ia: 0.8782 - loss: 0.0455 - mae: 0.1576 - rmse: 0.2014 - smape: 0.3508 - val_ia: 0.6180 - val_loss: 0.0059 - val_mae: 0.0546 - val_rmse: 0.0649 - val_smape: 0.1442

Epoch 3/128                                                                            

922/922 - 3s - 4ms/step - ia: 0.8937 - loss: 0.0366 - mae: 0.1381 - rmse: 0.1793 - smape: 0.3038 - val_ia: 0.6590 - val_loss: 0.0046 - val_mae: 0.0487 - val_rmse: 0.0584 - val_smape: 0.1545

Epoch 4/128                                                                            

922/922 - 3s - 3ms/step - ia: 0.9009 - loss: 0.0324 - mae: 0.1284 - rmse: 0.1682 - smape: 0.2800 - val_ia: 0.5853 - val_loss: 0.0058 - val_mae: 0.0603 - val_rms

In [16]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.0, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0005454046615408651, 'units': 4}
